# Review saved word-sentiment EEG results

Use a **CPU** runtime, choose **Runtime → Run all**, and allow Drive mounting. This notebook reads the completed `v4_word_erp_compact` results, verifies saved files, and creates subject-specific and scalp-region plots plus word-selection tables. No ZuCo extraction or source EEG download is performed. Code stays in the temporary Colab runtime.

The audit checks all 36 reader/task records, unique event IDs, label consistency, counts, arrays, reproduced grand averages, and the portable ZIP. It cannot independently validate the original EEG, acquisition units/montage, or the correctness of agent sentiment judgments. Figures remain descriptive fixation-related averages.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys
from IPython.display import Image, Markdown, display
drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/Thesis/Results/word_sentiment_eeg/v4_word_erp_compact')
if not (RUN / 'manifest.json').is_file():
    raise RuntimeError(f'Completed results were not found at {RUN}. Check the mounted Google account. No extraction will be started.')
REPO = Path('/content/word-sentiment-eeg-review')
def git(*args):
    subprocess.run(['git', '-c', 'credential.helper=', '-c', 'http.extraheader=', *map(str, args)], check=True, env=dict(os.environ, GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone', '--depth', '1', 'https://github.com/parmisbathayan/word-sentiment-eeg.git', REPO)
else:
    if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('The temporary review checkout has edits. Save them before updating it.')
    git('-C', REPO, 'pull', '--ff-only', 'origin', 'main')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'matplotlib'], check=True)
print('Code version:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip(), flush=True)
print('Reading existing results only:', RUN, flush=True)


In [ ]:
# Saved-file audit and subject figures; no EEG extraction.
sys.path.insert(0, str(REPO / 'scripts'))
import importlib, colab_progress
run_with_progress = importlib.reload(colab_progress).run_with_progress
command = [sys.executable, '-u', str(REPO / 'scripts/audit_subject_erps.py'), '--run-dir', str(RUN)]
run_with_progress(command, RUN / 'review_progress.json', activity='saved-file audit and subject plots')


In [ ]:
# Same saved trials, six fixed scalp partitions; no significance tests.
command = [sys.executable, '-u', str(REPO / 'scripts/review_regions.py'), '--run-dir', str(RUN)]
run_with_progress(command, RUN / 'region_review_progress.json', activity='saved-average scalp-region review')
display(Image(filename=str(RUN / 'plots/regions/channel_regions.png')))
for task in ['SR', 'NR', 'TSR']:
    display(Image(filename=str(RUN / f'plots/regions/{task}_regions.png')))


In [ ]:
import json
report = json.loads((RUN / 'audit/integrity.json').read_text())
display(Markdown(f"**Saved-file audit: {report['status']}** — {report['reader_task_records']} reader/task records; {report['accepted_epochs']:,} accepted epochs; {report['duplicate_event_ids']} duplicate event IDs."))
print('Audit limits:', '; '.join(report['not_checked']))
for task in ['SR', 'NR', 'TSR']:
    display(Image(filename=str(RUN / f'plots/subject_grids/{task}_subjects.png')))
print('One three-task figure per reader:', RUN / 'plots/subjects')
print('Descriptive window measurements:', RUN / 'audit/subject_window_means.csv')
display(Markdown((RUN / 'START_HERE.md').read_text()))


In [ ]:
# Inspect the words that actually contributed EEG, with their sentence context.
import pandas as pd
from ipywidgets import interact, Dropdown, Text
selected = pd.read_csv(RUN / 'audit/word_selection.csv', keep_default_na=False)
def show_words(task='all', sentiment='all', inclusion='accepted EEG', search=''):
    table = selected
    if task != 'all':
        table = table[table.task == task]
    if sentiment != 'all':
        table = table[table.contextual_class == sentiment]
    if inclusion == 'accepted EEG':
        table = table[table.accepted_eeg_observations > 0]
    if search.strip():
        table = table[table.word.str.contains(search, case=False, regex=False) | table.sentence.str.contains(search, case=False, regex=False)]
    print(f'{len(table):,} word/task rows match. Showing the first 50; CSV files contain all rows.')
    display(table[['task', 'word', 'sentence', 'lexical_class', 'contextual_class', 'label_eligible', 'accepted_eeg_observations', 'readers_with_accepted_eeg', 'annotation_note']].head(50))
interact(show_words, task=Dropdown(options=['all', 'SR', 'NR', 'TSR']), sentiment=Dropdown(options=['all', 'positive', 'negative', 'neutral', 'uncertain']), inclusion=Dropdown(options=['accepted EEG', 'all annotations']), search=Text(continuous_update=False))
print('Full word-selection table:', RUN / 'audit/word_selection.csv')
print('Words with accepted EEG:', RUN / 'audit/accepted_word_labels.csv')
